# Chapter 6: Design a Key-Value Store
*System Design Interview*

## TL;DR

A distributed key-value store partitions data via consistent hashing, replicates across N nodes, and uses tunable quorum (W/R/N) for consistency. The CAP theorem forces a choice between consistency and availability during partitions. Conflict resolution relies on vector clocks, failure detection on gossip protocol, and anti-entropy on Merkle trees.

## Requirements

| Requirement | Detail |
|---|---|
| API | `put(key, value)` and `get(key)` |
| Key-value size | < 10 KB per pair |
| High availability | Respond quickly even during failures |
| High scalability | Support large data sets |
| Automatic scaling | Add/remove servers based on traffic |
| Tunable consistency | Configurable via W, R, N |
| Low latency | Fast reads and writes |

## Estimation: Quorum Configurations

In [ ]:
# Explore quorum configurations and their guarantees
print('N  W  R  |  W+R>N?  |  Guarantee')
print('-' * 50)
configs = [
    (3, 1, 3, 'Fast write, slow read'),
    (3, 3, 1, 'Slow write, fast read'),
    (3, 2, 2, 'Balanced, strong consistency'),
    (3, 1, 1, 'Fastest, eventual consistency'),
    (5, 3, 3, 'Higher redundancy, strong'),
    (5, 2, 2, 'Higher redundancy, eventual'),
]
for n, w, r, label in configs:
    strong = 'YES' if w + r > n else 'NO '
    print(f'{n}  {w}  {r}  |  {strong}      |  {label}')

## Estimation: Replication and Availability

In [ ]:
# Availability with N replicas assuming independent failures
single_node_availability = 0.999  # 99.9% uptime

print('Replicas  |  P(all down)       |  System Availability')
print('-' * 58)
for n in [1, 2, 3, 5, 7]:
    p_all_down = (1 - single_node_availability) ** n
    system_avail = 1 - p_all_down
    nines = -1  # sentinel
    if system_avail < 1:
        import math
        nines = -math.log10(1 - system_avail)
    print(f'  N={n}     |  {p_all_down:.2e}  |  '
          f'{system_avail:.10f} ({nines:.1f} nines)')

## High-Level Design

```mermaid
graph TB
    Client --> Coordinator
    Coordinator --> S0[Node s0]
    Coordinator --> S1[Node s1]
    Coordinator --> S2[Node s2]

    subgraph Ring[Consistent Hash Ring]
        S0 --> S1
        S1 --> S2
        S2 --> S3[Node s3]
        S3 --> S0
    end

    S0 -.-> R0[Replica]
    S1 -.-> R1[Replica]
    S2 -.-> R2[Replica]
```

## Deep Dive

### CAP Theorem
- **CP** (Consistency + Partition Tolerance): Block writes during partition (e.g., bank systems)
- **AP** (Availability + Partition Tolerance): Accept writes, reconcile later (e.g., social feeds)
- **CA** does not exist in practice -- network partitions are unavoidable

### Data Partitioning
Consistent hashing distributes keys across nodes on a hash ring. Virtual nodes handle heterogeneous server capacities.

### Data Replication
Walk clockwise from a key's position to choose N unique physical servers. Place replicas in **distinct data centers** for disaster tolerance.

### Quorum Consensus
- `W + R > N` guarantees strong consistency (overlapping read/write sets)
- Common: `N=3, W=2, R=2`
- Fast read: `W=N, R=1`; Fast write: `W=1, R=N`

### Vector Clocks
Each write creates a `D([Si, vi], ...)` version vector. Conflict exists when no version dominates (some counters higher, some lower). Client resolves conflicts.

### Gossip Protocol
Nodes exchange heartbeat lists with random peers. If a heartbeat counter stalls past a threshold, the node is marked down.

### Failure Handling
- **Temporary**: Sloppy quorum + hinted handoff (healthy stand-in returns data when failed node recovers)
- **Permanent**: Merkle tree anti-entropy syncs only divergent data buckets

### Write Path
1. Persist to commit log
2. Save to memory cache
3. Flush to SSTable on disk when cache full

### Read Path
1. Check memory cache
2. If miss, consult bloom filter to find candidate SSTables
3. Read from SSTable and return

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Consistency vs Availability | CP: block writes during partition | AP: allow stale reads |
| W+R > N (strong) | Guaranteed fresh reads | Higher write latency |
| W+R <= N (eventual) | Low latency | Possible stale reads |
| Vector clock length | Accurate ancestry tracking | Memory grows with servers |
| Sloppy quorum | Higher availability | Temporary inconsistency |

## Key Takeaways

1. The CAP theorem forces an **explicit trade-off** -- know your system's priority
2. Quorum consensus (N/W/R) is the main **consistency tuning knob**
3. Vector clocks resolve concurrent-write conflicts in eventually consistent systems
4. Gossip protocol decentralizes failure detection without a single monitor
5. Merkle trees minimize data transfer during anti-entropy repair
6. Write path: commit log -> memory -> SSTable; Read path: memory -> bloom filter -> SSTable

## Related Concepts

- [[cap-theorem]] -- the fundamental distributed systems constraint
- [[data-partitioning]] -- consistent hashing for data distribution
- [[data-replication]] -- replicating across N nodes
- [[quorum-consensus]] -- tuning N, W, R for consistency/latency
- [[vector-clocks]] -- conflict detection and versioning
- [[gossip-protocol]] -- decentralized failure detection
- [[merkle-trees]] -- efficient anti-entropy synchronization